## Hent data

- BVP dataframes har 64 observationer/sek
- EDA dataframes har 4 observationer/sek
- HR dataframes har 1 observationer/sek
- TEMP dataframes har 4 observationer/sek

### Vi vælger at interpollere og downscale til 16 punkter i sekundet for at beholde mere information fra BVP

- BVP: 64 -> 16 (mean på hvert 4 punkt)
- EDA: 4 -> 16 (interpoler 3 nye punkter mellem hvert punktsæt)
- HR: 1 -> 16 (interpoler 15 nye punkter mellem hvert sekund)
- TEMP: 4 -> 16 (interpoler 15 nye punkter mellem hvert punktsæt)

In [1]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

In [26]:
def prepare_dataframe(path_to_phase, file):
    colname = file.split('.')[0]
    df = pd.read_csv(path_to_phase+'/'+ file)
    if df.isnull().any().any()==True:
        print(f'{df.isnull().sum()} missing values in {path_to_phase} {file} ')

    df['time'] = pd.to_datetime(df['time'], format='mixed')
    df = df.set_index('time').sort_index()
    intervals = df.index.to_series().diff().dropna()
    only_one_timeinterval = intervals.nunique() == 1
    if not only_one_timeinterval:
        print(f"Amount unique time intervals {intervals.nunique()} at {intervals}")

    if file== 'BVP.csv':
        df_downsampled = df.resample("62.5ms").mean()
        return df_downsampled[[colname]]
    else: 
        #sæt en interpoleringsfrekvens på 62.5 ms som svarer til 1/16 sekund og interpoler
        df_upsampled = df.resample("62.5ms").asfreq()
        df_upsampled[colname] = df_upsampled[colname].interpolate(method='linear')
        return df_upsampled[[colname]]
    
    
def create_phase_dataframe(path_to_phase):
    filenames = ['BVP.csv', 'EDA.csv', 'HR.csv', 'TEMP.csv']
    dfs_to_combine = []
    for file in filenames:
        df = prepare_dataframe(path_to_phase, file)
        dfs_to_combine.append(df)
    
    Collected_dataframes = pd.concat(dfs_to_combine, axis=1, join='inner')

    # 3. Hvis du gerne vil have 'time' tilbage som en almindelig kolonne til sidst:
    Collected_dataframes = Collected_dataframes.reset_index()
    return Collected_dataframes
        


In [22]:
import pandas as pd
import os


def append_to_csv(df, filename, path_to_folder):
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)
    filepath = os.path.join(path_to_folder,filename )
    # Check if file exists right now
    file_exists = os.path.isfile(filepath)
    
    # mode='a' : Append to the end of the file
    # header=not file_exists : If file exists, header is False. If not, header is True.
    df.to_csv(filepath, mode='a', index=False, header=not file_exists)
    

def create_resonse_dataframe(path_to_phase, round_=1, phase=1):
    df = pd.read_csv(os.path.join(path_to_phase, 'response.csv'))
    if df.isnull().any().any()==True:
        missing_cols = df.columns[df.isnull().any()]
        print(f'{df.isnull().sum().sum()} missing values in {path_to_phase} responses cols: {(", ").join(missing_cols)}')
    df = df.reset_index(drop=True)
    df = df.drop(columns=['index', 'Unnamed: 0'], errors='ignore')
    df.insert(0,column='Round', value = round_+1)
    df.insert(1,column='phase', value = phase+1) 
    return df

def define_phases_start_and_end(person_dict,phase_df ,r,p):
    person_dict[f'Round_{r+1}'][f'Phase_{p+1}'] = {}
    person_dict[f'Round_{r+1}'][f'Phase_{p+1}']['start_time'] = phase_df['time'][0]
    person_dict[f'Round_{r+1}'][f'Phase_{p+1}']['end_time'] = phase_df['time'][len(phase_df)-1]
    return person_dict

def initialize_person_dict(person_ID_and_group, gruppe, person, team_info):
    person_dict = {}
    person_dict['name'] = person_ID_and_group
    person_dict['day'] = gruppe
    person_dict['person_ID'] = person

    # Define team info
    mask_team_info = team_info['ID'] == person
    team = int(team_info[mask_team_info]['Team'].iloc[0])
    team_and_day = f"{gruppe}_{team}"
    Puzzler  = int(team_info[mask_team_info]['Puzzler'].iloc[0])
    person_dict['team'] = team_and_day
    person_dict['puzzler'] = f"{Puzzler}"

    return person_dict

import pickle
def save_dictionary(person_dict,path_to_folder,filename):
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)
    with open(os.path.join(path_to_folder,filename), 'wb') as handle:
        pickle.dump(person_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [20]:
(', ').join(missing_cols)

'determined, attentive, afraid, active'

In [27]:
import pandas as pd
import pathlib
import os
import pickle
def create_collected_datafile_all_participans(dataset_path='../data/raw/dataset'):

    # Definer stien
    #dataset_path = r'/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset'

    path_to_processed_folder = '../data/preprocessed'
    path_to_processed_biosignal = os.path.join(path_to_processed_folder, 'biosignal_mat')
    path_to_processed_responses = os.path.join(path_to_processed_folder, 'responses_mat')
    path_to_processed_info_dict = os.path.join(path_to_processed_folder, 'info_dicts')

    person_nummer = 1
    for gruppe in sorted(os.listdir(dataset_path)):
        gruppe_sti = os.path.join(dataset_path, gruppe)
        if not os.path.isdir(gruppe_sti): continue
        team_info = pd.read_csv(os.path.join(gruppe_sti,'team_info.csv'))
        
        # 2. Loop: Personer
        for person in sorted(os.listdir(gruppe_sti)):
            person_sti = os.path.join(gruppe_sti, person)
            person_ID_and_group = f'Person{str(person_nummer)}_{gruppe}_{person}'
            
            filename_person = person_ID_and_group+'.csv'
            filename_person_responses = person_ID_and_group+'_responses'+'.csv'
            filename_dictionary = person_ID_and_group+'_dict.pickle'

            # Hvis koden er kørt før skal personfilerne slettes og laves på ny. 
            # Ellers appender vi nye dataframes i forlængelse af de gamle
            if os.path.exists(os.path.join(path_to_processed_biosignal,filename_person)):
                os.remove(os.path.join(path_to_processed_biosignal,filename_person))
            if os.path.exists(os.path.join(path_to_processed_responses,filename_person_responses)):
                os.remove(os.path.join(path_to_processed_responses,filename_person_responses))
            if not os.path.isdir(person_sti): continue
            person_dict = initialize_person_dict(person_ID_and_group, gruppe, person,team_info)
        

            person_nummer +=1
            
            # 3. Loop: Runder
            for r,runde in enumerate(sorted(os.listdir(person_sti))):
                runde_sti = os.path.join(person_sti, runde)
                if not os.path.isdir(runde_sti): continue
                person_dict[f'Round_{r+1}'] ={}
                
                # 4. Loop: Phase
                for p, phase in enumerate(sorted(os.listdir(runde_sti))):
                    phase_sti = os.path.join(runde_sti, phase)
                    #print(phase_sti)
                    if not os.path.isdir(phase_sti): continue
                    phase_df = create_phase_dataframe(phase_sti)
                    append_to_csv(phase_df, filename_person,path_to_processed_biosignal)
                    response_df = create_resonse_dataframe(phase_sti, round_=r, phase=p)
                    append_to_csv(response_df, filename_person_responses,path_to_processed_responses)

                    # append start and end times to dict
                    person_dict = define_phases_start_and_end(person_dict,phase_df ,r,p)
                    if p==0:
                        person_dict[f'Round_{r+1}']['start_time'] = phase_df['time'][0]
                person_dict[f'Round_{r+1}']['end_time'] = phase_df['time'][len(phase_df)-1]
        
            # save dictionary
            save_dictionary(person_dict,path_to_processed_info_dict,filename_dictionary)
                    

create_collected_datafile_all_participans()

4 missing values in ../data/raw/dataset/D1_1/ID_1/round_4/phase1 responses cols: determined, attentive, afraid, active
1 missing values in ../data/raw/dataset/D1_1/ID_3/round_2/phase1 responses cols: inspired
1 missing values in ../data/raw/dataset/D1_1/ID_3/round_3/phase1 responses cols: inspired
1 missing values in ../data/raw/dataset/D1_1/ID_3/round_3/phase3 responses cols: determined
1 missing values in ../data/raw/dataset/D1_1/ID_3/round_4/phase2 responses cols: difficulty
1 missing values in ../data/raw/dataset/D1_3/ID_1/round_1/phase3 responses cols: difficulty
1 missing values in ../data/raw/dataset/D1_3/ID_1/round_2/phase1 responses cols: difficulty
1 missing values in ../data/raw/dataset/D1_3/ID_1/round_2/phase3 responses cols: difficulty
1 missing values in ../data/raw/dataset/D1_3/ID_1/round_3/phase1 responses cols: difficulty
1 missing values in ../data/raw/dataset/D1_3/ID_1/round_3/phase3 responses cols: difficulty
1 missing values in ../data/raw/dataset/D1_3/ID_2/round_1

In [3]:
21/28

0.75

In [ ]:
df = pd.read_csv('/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_1/round_4/phase1/response.csv')
df

{'name': 'Person6_D1_1_ID_6',
 'day': 'D1_1',
 'person_ID': 'ID_6',
 'team': 'D1_1_4',
 'puzzler': '0',
 'Round_1': {'Phase_1': {'start_time': Timestamp('2021-12-17 16:11:55'),
   'end_time': Timestamp('2021-12-17 16:19:22')},
  'start_time': Timestamp('2021-12-17 16:11:55'),
  'Phase_2': {'start_time': Timestamp('2021-12-17 16:24:54'),
   'end_time': Timestamp('2021-12-17 16:30:40')},
  'Phase_3': {'start_time': Timestamp('2021-12-17 16:33:20'),
   'end_time': Timestamp('2021-12-17 16:38:18')},
  'end_time': Timestamp('2021-12-17 16:38:18')},
 'Round_2': {'Phase_1': {'start_time': Timestamp('2021-12-17 16:42:23'),
   'end_time': Timestamp('2021-12-17 16:47:28')},
  'start_time': Timestamp('2021-12-17 16:42:23'),
  'Phase_2': {'start_time': Timestamp('2021-12-17 16:49:36'),
   'end_time': Timestamp('2021-12-17 16:54:53')},
  'Phase_3': {'start_time': Timestamp('2021-12-17 16:56:39'),
   'end_time': Timestamp('2021-12-17 17:02:03')},
  'end_time': Timestamp('2021-12-17 17:02:03')},
 'Ro

In [ ]:
# person_nummer = 1
# person_sti = '/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_1'
# person_ID_and_group = f'Person{str(person_nummer)}_D1_1_ID_1'
# gruppe = 'D1_1'
# person = 'ID_1'

# path_to_processed_folder = '../data/preprocessed'
# path_to_processed_biosignal = os.path.join(path_to_processed_folder, 'biosignal')
# path_to_processed_responses = os.path.join(path_to_processed_folder, 'responses')
# person_dict = initialize_person_dict(person_ID_and_group, gruppe, person)


# person_nummer +=1
# filename_person_responses = person_ID_and_group+'_responses'+'.csv'
# filename_person = person_ID_and_group+'.csv'
# if os.path.exists(os.path.join('../data/preprocessed/responses',filename_person_responses)):
#     os.remove(os.path.join('../data/preprocessed/responses',filename_person_responses))
# for r,runde in enumerate(sorted(os.listdir(person_sti))):
#     runde_sti = os.path.join(person_sti, runde)
#     if not os.path.isdir(runde_sti): continue
#     person_dict[f'Round_{r+1}'] = {}
    
#     # 4. Loop: Phaser
#     for p, phase in enumerate(sorted(os.listdir(runde_sti))):
#         phase_sti = os.path.join(runde_sti, phase)
#         #print(phase_sti)
#         if not os.path.isdir(phase_sti): continue
#         phase_df = create_phase_dataframe(phase_sti)
#         append_to_csv(phase_df, filename_person,path_to_processed_biosignal)
#         response_df = create_resonse_dataframe(phase_sti, round_=r, phase=p)
#         append_to_csv(response_df, filename_person_responses,path_to_processed_responses)
#         # Update dictionary
#         person_dict = define_phases_start_and_end(person_dict,phase_df ,r,p)

#         if p==0:
#             person_dict[f'Round_{r+1}']['start_time'] = phase_df['time'][0]
#     person_dict[f'Round_{r+1}']['end_time'] = phase_df['time'][len(phase_df)-1]

#     print(f'round {runde} done')



round round_1 done
round round_2 done
round round_3 done
round round_4 done


In [6]:
df = pd.read_csv('/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_1/round_4/phase1/response.csv')
df

,Unnamed: 0,particpant_ID,puzzler,team_ID,E4_nr,upset,hostile,alert,ashamed,inspired,nervous,determined,attentive,afraid,active,frustrated
0,0,7,1,2,A0388C,1,1,1,1,2,1,NaN,NaN,NaN,NaN,3


In [16]:
missing_cols = df.columns[df.isnull().any()]
missing_cols

Index(['determined', 'attentive', 'afraid', 'active'], dtype='str')

In [24]:
df = pd.read_csv('/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_3/round_2/phase2/BVP.csv')
df.head()

,Unnamed: 0,BVP,time
0,0,-86.00,2021-12-17 16:49:37.125000
1,1,-110.83,2021-12-17 16:49:37.140625
2,2,-130.02,2021-12-17 16:49:37.156250
3,3,-146.12,2021-12-17 16:49:37.171875
4,4,-162.93,2021-12-17 16:49:37.187500


In [ ]:
file = 'BVP.csv'
colname = 'BVP'
df = pd.read_csv('/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_3/round_2/phase2/BVP.csv')

# if df.isnull().any().any()==True:
#     print(f'{df.isnull().sum()} missing values in {path_to_phase} {file} ')

df['time'] = pd.to_datetime(df['time'], format='mixed')
df = df.set_index('time').sort_index()

intervals = df.index.to_series().diff().dropna()
only_one_timeinterval = intervals.nunique() == 1
if not only_one_timeinterval:
    print(f"Amount unique time intervals {intervals.nunique()} at {intervals}")

print(intervals.nunique() == 1)
print(intervals.iloc[0])


True
0 days 00:00:00.015625
